# Kvasir-VQA x1 — BLIP-2 VQA fine-tuning

Fine-tune a stronger VQA transformer (BLIP-2 / InstructBLIP) on the x1 splits. The notebook mirrors the BLIP baseline but upgrades the backbone and keeps outputs under `2_modeling/06_blip2_finetune/out/`.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from transformers import TrainingArguments, Trainer
import evaluate

2026-01-14 09:02:22.834198: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-14 09:02:22.834233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-14 09:02:22.835384: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-14 09:02:22.841840: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-14 09:02:23.732421: W tensorflow/compiler/tf2

In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "06_blip2_finetune" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Swap to another backbone if you prefer (e.g., 'Salesforce/instructblip-vicuna-7b' for instruction-tuned VQA)
MODEL_NAME = "Salesforce/blip2-flan-t5-xl"
PROMPT_TEMPLATE = "Question: {question}\nAnswer:"

SEED = 42
MAX_ANSWER_LEN = 16
QUESTION_MAX_LEN = 96
MAX_GEN_TOKENS = 16

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LR = 1e-5

MAX_TRAIN_SAMPLES = None  # set int for smoke tests
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/06_blip2_finetune/out
Device: cuda


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Optional subsampling
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES is not None:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES is not None:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

{'train': 46966, 'val': 5931, 'test': 5952}


In [5]:
# Load processor + model
processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.to(DEVICE)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
model.config.text_config.pad_token_id = processor.tokenizer.pad_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.use_cache = False


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 100.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 58.19 MiB is free. Process 631189 has 6.60 GiB memory in use. Process 2192693 has 340.00 MiB memory in use. Including non-PyTorch memory, this process has 7.54 GiB memory in use. Of the allocated memory 7.23 GiB is allocated by PyTorch, and 96.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
        inputs = processor(images=image, text=prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=QUESTION_MAX_LEN)
        labels = processor.tokenizer(
            str(row["answer"]) if pd.notna(row["answer"]) else "",
            max_length=MAX_ANSWER_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == processor.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = labels
        return item

def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

train_ds = VQADataset(train_df)
val_ds = VQADataset(val_df)
test_ds = VQADataset(test_df)

print("Sample prompt:", PROMPT_TEMPLATE.format(question=train_df.iloc[0]["question"]))

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print(trainer)


In [ ]:
# Train
trainer.train()

# Save final model
final_dir = OUT_DIR / "final_model"
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
print("Saved model to", final_dir)

In [ ]:
# Evaluate with BLEU/ROUGE on test split
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

model.eval()

def generate_answer(row):
    img = Image.open(row["image_path"]).convert("RGB")
    prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_GEN_TOKENS)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="BLIP-2 eval"):
    preds.append(generate_answer(row))
    refs.append(str(row["answer"]))

bleu_refs = [[r] for r in refs]
bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]

results = {"bleu": bleu_score, "rougeL": rouge_l}
with open(OUT_DIR / "metrics_test.json", "w") as f:
    json.dump(results, f, indent=2)

print(results)